# Diagnóstico de cumplimiento logístico — 1S 2026

**Encargo:** el reporte mensual muestra OTIF > 95%, pero dirección percibe que
"las entregas se sienten más lentas". Entregar: (1) una definición defendible del
KPI de cumplimiento, (2) los cuellos de botella con evidencia, (3) el costo de no
actuar.

**Insumos:** `/envios.csv` (~62 mil envíos, ene–jun 2026, timestamps de cada
etapa del ciclo) y `/tiendas.csv`. *Datos sintéticos generados para este caso
de estudio (ver `simulador/`); sin afiliación con ninguna empresa.*

**Ruta del análisis:** auditoría → limpieza con bitácora → definición del KPI →
segmentación → validación estadística → cuantificación en pesos.

## Preparación del data frame (DF)

1. `parse_dates=TS` le dice a pandas que esas columnas son fechas y las convierta a tipo `datetime64`
2. valores faltantes en fechas se llaman `NaT`

In [27]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

TS = ["ts_orden", "ts_surtido_cedis", "ts_llegada_destino",
      "ts_disponible_cliente", "ts_entrega_final", "ts_promesa_entrega"]

env = pd.read_csv("datos/envios.csv", parse_dates=TS)
tie = pd.read_csv("datos/tiendas.csv")

## Auditoría de datos

1. `df.shape` → tupla (filas, columnas). `df.dtypes` → tipo de cada columna.
2. `df.col.duplicated()` devuelve una Serie booleana (True donde el valor ya apareció antes). `.sum()` sobre booleanos cuenta los True, porque True==1.
3. `value_counts()` = histograma de una columna categórica. Detector de inconsistencias: "Softline", "SOFTLINE" y "soft line" aparecen como categorías separadas.
4. Sanity checks: busca datos físicamente imposibles. `env.valor <= 0` es booleano y `.sum()` genera el conteo de valores absurdos. `env.piezas == 0` son envíos fantasma y `env.entrega < env.orden` puede deberse a problemas de husos horarios, errores manuales o algún error en escáneres de los repartidores.

In [28]:
env.head()

,envio_id,orden_id,sku,categoria,es_big_ticket,cadena,canal,zona,cedis_origen,tienda_destino,transportista,piezas,valor_mercancia_mxn,costo_envio_mxn,ts_orden,ts_surtido_cedis,ts_llegada_destino,ts_disponible_cliente,ts_entrega_final,ts_promesa_entrega,estado_final
0,E2026005275,O39563936,SKU-703689,Electronica,True,Liverpool,Tienda a Tienda,Norte,CD-MONTERREY,T1032,PaqueteExpress,1,14225.66,318.22,2026-02-22 12:24:15,2026-02-23 00:55:41,2026-02-23 14:07:36,2026-02-23 18:29:39,2026-02-23 18:29:39,2026-02-28 20:00:00,Entregado
1,E2026028446,O18074941,SKU-396219,Hardline,False,Suburbia,Click & Collect,Metro,CD-TULTITLAN,T1004,LogiMex,5,974.51,57.78,2026-04-08 08:46:33,2026-04-08 21:18:54,2026-04-09 04:38:00,2026-04-09 06:29:42,2026-04-09 17:56:15,2026-04-10 20:00:00,Entregado
2,E2026032896,O62313221,SKU-842869,Muebles,True,Liverpool,Domicilio,Occidente,CD-GUADALAJARA,T1027,EnviaYa,2,23990.88,1121.65,2026-02-12 19:26:00,2026-02-13 05:32:34,2026-02-13 21:16:02,2026-02-14 18:21:54,2026-02-14 18:21:54,2026-02-17 20:00:00,Entregado
3,E2026017709,O75576285,SKU-864873,Linea Blanca,True,Liverpool,Domicilio,Norte,CD-MONTERREY,T1036,LogiMex,2,14636.78,410.55,2026-02-11 08:55:35,2026-02-12 03:12:01,2026-02-12 19:01:59,2026-02-14 23:10:53,2026-02-14 23:10:53,2026-02-17 20:00:00,Entregado
4,E2026060785,O92395979,SKU-346153,Softline,False,Liverpool,Click & Collect,Metro,CD-TULTITLAN,T1015,EnviaYa,1,906.46,66.74,2026-04-20 22:26:24,2026-04-21 04:07:31,2026-04-21 23:34:34,2026-04-21 23:58:51,2026-04-18 22:26:24,2026-04-22 20:00:00,Entregado


In [29]:
# Listado de columnas y cuántos vacíos tiene cada una.
print(env.isna().sum())

envio_id                    0
orden_id                    0
sku                         0
categoria                   0
es_big_ticket               0
cadena                      0
canal                       0
zona                        0
cedis_origen                0
tienda_destino              0
transportista               0
piezas                      0
valor_mercancia_mxn         0
costo_envio_mxn             0
ts_orden                    0
ts_surtido_cedis          758
ts_llegada_destino       2423
ts_disponible_cliente    2345
ts_entrega_final         2342
ts_promesa_entrega          0
estado_final                0
dtype: int64


In [30]:
# ¿costos de envío negativos o en cero? (Quizá un error de facturación)
print((env.costo_envio_mxn <= 0).sum())

# ¿algún paquete marcado como Big Ticket pero que cuesta muy poco?
# (Cruzar lógica booleana para encontrar contradicciones)
# print(((env.es_big_ticket == True) & (env.valor_mercancia_mxn < 500)).sum())
# opcional, requiere alguna métrica establecida de "poco" o "mucho" para peso y valor de mercancía.

0


In [31]:
# Un ciclo 'for' para auditar rápidamente las columnas de texto clave
columnas_texto = ['estado_final', 'canal', 'cadena', 'zona']

for col in columnas_texto:
    print(f"\n--- Auditoría de: {col} ---")
    print(env[col].value_counts(dropna=False))
    # dropna=False: si hay nulos ocultos en el texto los mostrará en el conteo como NaN.


--- Auditoría de: estado_final ---
estado_final
Entregado    55839
Devuelto      4236
Cancelado     2345
Name: count, dtype: int64

--- Auditoría de: canal ---
canal
Domicilio          34646
Click & Collect    18112
Tienda a Tienda     9662
Name: count, dtype: int64

--- Auditoría de: cadena ---
cadena
Liverpool    41920
Suburbia     20500
Name: count, dtype: int64

--- Auditoría de: zona ---
zona
Metro        23503
Norte        12537
Occidente     9954
Sureste       8755
Bajio         7671
Name: count, dtype: int64


In [32]:
print(env.shape)                                    # dimensiones del crudo
print(env.envio_id.duplicated().sum())              # duplicados de llave
print(env.categoria.value_counts().head(12))        # variantes sucias
print(env[TS].isna().sum())                         # nulos por timestamp
print((env.valor_mercancia_mxn <= 0).sum())         # valores imposibles
print((env.piezas == 0).sum())                      # envíos fantasma
print((env.ts_entrega_final < env.ts_orden).sum())  # secuencias imposibles

(62420, 21)
420
categoria
Softline        19280
Hardline         8978
Electronica      7900
Belleza          6793
Linea Blanca     5600
Muebles          4361
Juguetes         3441
SOFTLINE         1251
soft line         838
HARDLINE          569
ELECTRONICA       488
BELLEZA           457
Name: count, dtype: int64
ts_orden                    0
ts_surtido_cedis          758
ts_llegada_destino       2423
ts_disponible_cliente    2345
ts_entrega_final         2342
ts_promesa_entrega          0
dtype: int64
278
181
60


**Lectura de los nulos.** Hay 2,325 pedidos en estado Cancelado y 2,345 nulos en
`ts_disponible_cliente`: los cancelados explican casi todo el hueco,
un pedido cancelado nunca llega a estar disponible. La columna `ts_llegada_destino`
tiene 2,423 nulos; descontando los cancelados que se cortaron antes de llegar,
el resto podrían ser devoluciones a origen, paquetes perdidos en tránsito o fallas
de captura del escáner (hipótesis a validar). Los 758 nulos
de `ts_surtido_cedis` apuntan sobre todo a fallas de captura en almacén: el porcentaje de
captura incompleta por etapa se propone como **métrica de calidad de dato** para
el tablero.

## Limpieza (linaje de dato)

Todo cambio se registra en el log

In [33]:
# Hay que cuidarse de los duplicados, porque no todos son clones perfectos. Algunos podrían tener información contradictoria.
# Por ejemplo, un mismo envío podría tener dos fechas de entrega final distintas.
# Contamos cuántos IDs se repiten
dups_por_id = env.duplicated(subset="envio_id").sum()

# Contamos cuántas filas son clones perfectos (todas las columnas iguales)
dups_exactos = env.duplicated().sum()

print(f"Duplicados por ID: {dups_por_id}")
print(f"Duplicados exactos: {dups_exactos}")

# Aquí coinciden (420 = 420): son clones perfectos y da igual con cuál quedarse.
# Si hubiera conflicto, ordenaríamos por la fecha de última actualización antes de deduplicar:
# env = env.sort_values(by="ts_llegada_destino", ascending=False)
# y keep="first" se quedaría con el registro más reciente.

Duplicados por ID: 420
Duplicados exactos: 420


In [34]:
# Creamos un diccionario para llevar un registro de los cambios que vamos haciendo (Data Cleaning Log)
log = {"crudo": len(env)}

# (1) Duplicados exactos de la llave: nos quedamos con la primera aparición
env = env.drop_duplicates(subset="envio_id", keep="first")
log["sin duplicados"] = len(env)

# (2) Normalizar texto: quitar espacios, unificar mayúsculas y variantes
env["categoria"] = (env.categoria.str.strip().str.lower()
                    .str.replace("soft line", "softline", regex=False)
                    .str.title())
env["transportista"] = env.transportista.str.strip()

# (3) Secuencias temporales imposibles: la fila no es confiable, fuera
mal = (env.ts_entrega_final < env.ts_orden) | (env.ts_surtido_cedis < env.ts_orden)  # boolean mask buscando secuencias imposibles
# luego, sumamos cuántas filas cumplen la condición mal y lo guardamos en el log. son true o false, así que sum() nos da el total de true.
log["secuencia imposible"] = int(mal.sum())
# si hay algún NaT la comparación devuelve NA. fillna(False) convierte esos NA en False, para que no se borre la fila.
# y luego invertimos la máscara con ~ para quedarnos con las filas que no son malas.
env = env[~mal.fillna(False)]

# (4) Valores numéricos imposibles: se anulan (NaN), la fila se queda
env.loc[env.valor_mercancia_mxn <= 0, "valor_mercancia_mxn"] = np.nan
env.loc[env.piezas <= 0, "piezas"] = np.nan

# Bitácora final: los cancelados existen pero no entran al denominador del KPI
log["cancelados (fuera del KPI)"] = int((env.estado_final == "Cancelado").sum())
log["base para KPI"] = int((env.estado_final != "Cancelado").sum())
for paso, filas in log.items():
    print(f"{paso:<28} {filas:>8,}")

crudo                          62,420
sin duplicados                 62,000
secuencia imposible                60
cancelados (fuera del KPI)      2,325
base para KPI                  59,615


## Definir el KPI

In [35]:
H = lambda a, b: (a - b).dt.total_seconds() / 3600
# Timedelta en horas entre dos timestamps. H(a,b) = a-b en horas.

# Creamos nuevas columnas de tiempo en horas, para poder hacer análisis de tiempos de entrega y logística.
env["mes"]            = env.ts_orden.dt.to_period("M").astype(str)  # Periodo mensual, trunca al mes.
env["h_surtido"]      = H(env.ts_surtido_cedis, env.ts_orden)
env["h_transito"]     = H(env.ts_llegada_destino, env.ts_surtido_cedis)
env["h_ultima_milla"] = H(env.ts_disponible_cliente, env.ts_llegada_destino)
env["h_ciclo"]        = H(env.ts_disponible_cliente, env.ts_orden)
env["h_recoleccion"]  = H(env.ts_entrega_final, env.ts_disponible_cliente)

In [36]:
# Separamos el KPI de OTIF en dos partes: operativo y percibido. El primero es la promesa de entrega, el segundo es la percepción del cliente.
env["otif_operativo"] = env.ts_disponible_cliente <= env.ts_promesa_entrega
env["otif_percibido"] = env.ts_entrega_final      <= env.ts_promesa_entrega

# Para el KPI de OTIF, nos quedamos con los envíos que no fueron cancelados. Los cancelados no cuentan para el KPI.
k = env[env.estado_final != "Cancelado"].copy()          # base del KPI

# Dividimos por canal y calculamos el promedio de OTIF operativo y percibido.
k.groupby("canal")[["otif_operativo", "otif_percibido"]].mean().round(4)

,otif_operativo,otif_percibido
canal,,
Click & Collect,0.9070,0.5519
Domicilio,0.9741,0.9741
Tienda a Tienda,0.9901,0.9901


En Domicilio y Tienda a Tienda las dos definiciones
coinciden (el evento es el mismo). En Click & Collect se abren ~35 puntos: la
operación ya había dejado el pedido disponible y el resto es la agenda del
cliente. Medir el OTIF logístico con `ts_entrega_final` castigaría a logística
por algo que no controla → **el KPI logístico corta en "disponible para el
cliente"**; el tiempo de recolección se reporta aparte, como métrica de
tienda/experiencia.

In [37]:
k.otif_operativo.mean()

0.9569235930554391

In [38]:
k.groupby("mes").otif_operativo.mean().round(4)
# en mayo hubo un ligero descenso en el OTIF operativo, que logra recuperarse en junio.
# Esto puede deberse a factores estacionales, cambios en la logística o problemas puntuales con transportistas.
# El global cumple meta: si hay un problema, está escondido debajo del agregado → segmentar.

mes
2026-01    0.9641
2026-02    0.9606
2026-03    0.9649
2026-04    0.9604
2026-05    0.9415
2026-06    0.9606
Name: otif_operativo, dtype: float64

## Segmentación

In [39]:
# Parte el DataFrame en grupos por una(s) columna(s), aplica una agregación a cada grupo, junta los resultados
k.groupby("canal").h_ciclo.agg(mediana="median",
                               p90=lambda s: s.quantile(0.90),
                               media="mean").round(1)
# .agg permite aplicar varias funciones de agregación a la vez, y renombrar las columnas resultantes.
# para percentiles, usamos una función lambda que calcula el quantile deseado.
# media > mediana en los tres canales: cola derecha; se reporta mediana y P90, no promedio.

,mediana,p90,media
canal,,,
Click & Collect,34.5,62.7,38.4
Domicilio,50.0,84.1,54.1
Tienda a Tienda,36.4,65.0,40.1


In [40]:
# groupby en 2D: una dimensión a filas, otra a columnas. Ideal para "métrica por X y por mes"
# .pivot_table permite crear una tabla dinámica: mediana del tiempo de surtido por CEDIS de origen y por mes.
k.pivot_table(index="cedis_origen", columns="mes",
              values="h_surtido", aggfunc="median").round(1)

mes,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
cedis_origen,,,,,,
CD-GUADALAJARA,8.3,7.9,8.1,8.2,10.9,8.0
CD-HUEHUETOCA,8.1,8.0,8.2,8.1,11.1,8.1
CD-MONTERREY,8.2,8.4,8.3,8.0,11.3,8.3
CD-PUEBLA,8.3,7.9,8.1,8.2,10.9,8.3
CD-TULTITLAN,8.2,8.3,8.5,9.2,16.1,14.3


**Hallazgo 1: CD-Tultitlán.** En mayo TODOS suben de ~8 a ~11 h. Pico ocasionado por evento tipo
Hot Sale: más volumen, colas más largas. Estacional. Tultitlán ya venía
separándose en abril (9.2 vs 8.1): señal de problema propio. En junio los demás
regresan a ~8; Tultitlán se queda en 14.3. **No era el volumen.** Los otros
cuatro CEDIS son grupo de control.

Deterioro estructural de capacidad de surtido.

In [41]:
# Ranking de peores tiendas por OTIF operativo, considerando solo aquellas con al menos 200 envíos.
# envios="size" es el conteo de envíos por tienda
# .sort_values("otif") ordena de menor a mayor, para ver las tiendas con peor OTIF primero.
st = k.groupby("tienda_destino").otif_operativo.agg(otif="mean", envios="size")
st[st.envios >= 200].sort_values("otif").head(10).round(4)

,otif,envios
tienda_destino,,
T1027,0.8485,924
T1026,0.8511,947
T1023,0.8522,981
T1030,0.8532,981
T1025,0.8537,964
T1031,0.8591,951
T1022,0.8599,921
T1029,0.8605,982
T1024,0.8610,935


In [42]:
# Las 10 peores comparten zona; sospecha de variable de confusión. Condicionamos por zona:
tz = k.groupby(["zona", "transportista"]).agg(envios=("envio_id", "size"),
                                              otif=("otif_operativo", "mean"))
# agrupamos por zona y transportista, calculando el número de envíos (size) y el OTIF operativo (promedio) para cada combinación.
zona = k.groupby("zona").otif_operativo.mean().rename("otif_zona")
# renombramos el mean de OTIF por zona como "otif_zona" para poder hacer el join con el DataFrame tz.
tz = tz.join(zona, on="zona")
tz["brecha_pts"] = (tz.otif - tz.otif_zona) * 100
# la brecha en puntos porcentuales entre el OTIF del transportista y el OTIF promedio de SU zona.
# Un valor negativo indica que el transportista está por debajo del promedio de la zona.
tz[tz.envios >= 200].sort_values("brecha_pts").head(3).round(4)
# ordenamos por brecha para ver los transportistas con peor desempeño relativo a su zona (mínimo 200 envíos).

,,envios,otif,otif_zona,brecha_pts
zona,transportista,,,,
Occidente,TransBajio,1481,0.6090,0.8564,-24.7390
Sureste,PaqueteExpress,1125,0.9458,0.9578,-1.2031
Norte,TransBajio,1861,0.9678,0.9738,-0.6048


**Hallazgo 2: TransBajio en Occidente.** 25 puntos por debajo de sus pares *de
la misma zona*. El mismo transportista en Norte opera normal por lo que el problema es su
operación regional. Se podría renegociar SLA / reasignar la ruta de esa zona.

In [43]:
cc = k[k.canal == "Click & Collect"]
rec = cc.groupby("tienda_destino").agg(envios=("envio_id", "size"),
                                       mediana_h=("h_recoleccion", "median"))
rec[rec.envios >= 100].sort_values("mediana_h", ascending=False).head(7).round(1)

,envios,mediana_h
tienda_destino,,
T1058,249,64.6
T1052,259,62.0
T1027,295,58.4
T1003,307,57.8
T1011,294,55.0
T1040,284,53.9
T1045,280,21.1


**Hallazgo 3:recolección en Click & Collect.** Seis tiendas triplican la
mediana de la red (~18 h) en el tramo "disponible to recogido". Como el corte
logístico ya se cumplió, el problema vive en la notificación al cliente o el
mostrador de entregas.

## Significancia

Con OTIF ≈ 95% los fallos son eventos raros y una tienda con pocos envíos trae barra de error
grande. Intervalo de Wilson (mejor que el intervalo normal cuando p está cerca de
0 o 1).

In [44]:
def wilson(k_exitos, n, z=1.96):
    p = k_exitos / n
    d = 1 + z**2/n
    centro = (p + z**2/(2*n)) / d
    ancho  = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return centro - ancho, centro + ancho

In [45]:
# Calculamos el promedio oficial (la barra a superar) de cada ZONA
otif_por_zona = k.groupby("zona")["otif_operativo"].mean().rename("otif_zona").reset_index()

# Agrupamos por TIENDA (asegurándonos de llevarnos la columna 'zona' también)
tiendas = k.groupby(["tienda_destino", "zona"]).agg(
    n=("envio_id", "size"),
    k_exitos=("otif_operativo", "sum"),
    otif_real=("otif_operativo", "mean")
).reset_index()

# Calculamos los Intervalos de Wilson (creamos las columnas ic_bajo y ic_alto)
tiendas[['ic_bajo', 'ic_alto']] = tiendas.apply(
    lambda row: pd.Series(wilson(row['k_exitos'], row['n'])), axis=1
)

# Unimos la meta de la zona con la tabla de las tiendas
tiendas = tiendas.merge(otif_por_zona, on="zona")

# Filtro: qué tiendas tienen su IC superior por debajo del OTIF de la zona.
tiendas["foco_rojo"] = tiendas.ic_alto < tiendas.otif_zona

culpables = tiendas[tiendas.foco_rojo]
print(culpables[["tienda_destino", "zona", "otif_real", "ic_alto", "otif_zona"]].round(4))
print(f"\nTiendas significativamente peores que su zona: {len(culpables)} de {len(tiendas)}")

   tienda_destino   zona  otif_real  ic_alto  otif_zona
54          T1054  Bajio     0.9481   0.9606     0.9659

Tiendas significativamente peores que su zona: 1 de 61


**Lectura.** El ranking crudo señalaba 10 tiendas; con intervalos de confianza y
comparando contra pares de su zona, sobrevive **1 de 61**. Paradoja de Simpson, la variable explicativa era el transportista de la zona, no las tiendas.

## Big Ticket

In [46]:
# Creamos la columna de exposición (Valor en Riesgo)
# Si el OTIF es Falso, el ~ lo vuelve Verdadero (1), y se multiplica por el valor.
# Si el OTIF es Verdadero (a tiempo), el ~ lo vuelve Falso (0), anulando el dinero.
k["valor_riesgo"] = k["valor_mercancia_mxn"] * (~k["otif_operativo"])

seg = np.where(k.es_big_ticket, "Big Ticket", "Resto")
# np.where crea un array de strings, donde cada fila se etiqueta como "Big Ticket" o "Resto" según la condición k.es_big_ticket.

k.groupby(seg).agg(envios=("envio_id", "size"),
                   otif=("otif_operativo", "mean"),
                   p90=("h_ciclo", lambda s: s.quantile(.9)),   # percentil 90 del tiempo de ciclo por segmento
                   valor=("valor_mercancia_mxn", "sum"),
                   riesgo=("valor_riesgo", "sum")).round(2)

,envios,otif,p90,valor,riesgo
Big Ticket,18472,0.98,90.74,3.289363e+08,7552565.43
Resto,41143,0.95,67.72,4.715109e+07,2455692.07


**Hallazgo 4:.** Big Ticket es ~31% de los envíos
pero ~87% del valor de mercancía; su OTIF es incluso mejor que el del resto y aun
así concentra ~75% del valor entregado fuera de promesa. Un punto de OTIF en Big
Ticket vale ~$3.3M de mercancía. Y su tiempo extra vive en surtido y última milla
— justo donde pegan los hallazgos 1 y 2: la mercancía más valiosa pasa por los
nodos rotos.

## Resumen ejecutivo

- **OTIF operativo semestre: 95.7% · P90 del ciclo: 77 h**
- **Valor entregado fuera de promesa: $10.0 millones MXN (75% en Big Ticket)**
- **Causa 1:** CD-Tultitlán degradado desde abril, sin recuperarse en junio (14.3 vs ~8 h de surtido)
- **Causa 2:** TransBajio en Occidente: OTIF 61% vs 88–91% de sus pares en la misma zona
- **Causa 3 (no logística):** 6 tiendas con recolección C&C al triple de la red

Acciones propuestas: redefinir el corte del OTIF logístico en
"disponible" y reportar recolección como métrica de tienda; diagnóstico de
capacidad de surtido en Tultitlán con datos del WMS; renegociar el SLA regional
de TransBajio priorizando la salida de Big Ticket.

In [47]:
resumen = {
    "otif_operativo": round(k.otif_operativo.mean(), 4),
    "p90_ciclo_h": round(k.h_ciclo.quantile(0.90), 1),
    "envios_base_kpi": len(k),
    "valor_fuera_de_promesa_mxn": round(k.valor_riesgo.sum(), 0),
}
for nombre, valor in resumen.items():
    print(f"{nombre:<28} {valor:>14,}")

otif_operativo                       0.9569
p90_ciclo_h                            77.0
envios_base_kpi                      59,615
valor_fuera_de_promesa_mxn     10,008,258.0
